In [1]:
# yolo4 object 검출
import os

import cv2
import numpy as np

# 파일에서 클래스명을 읽는 함수
def read_classes(file):
    classes = None
    with open(file, mode='r', encoding='utf-8') as f:
        classes = f.read().splitlines()
    return classes

# 클래스 수만큼 (랜덤으로) 컬러 테이블을 생성하는 함수 (객체를 사각형)
def get_colors(num):
    colors = []
    np.random.seed(0)
    for i in range(num):
       color = np.random.randint(0, 256, [3]).astype(np.uint8)
       colors.append(color.tolist())
    return colors

In [2]:
# image load
#capture = cv2.VideoCapture("dog.jpg")

# camera
capture = cv2.VideoCapture(0)

if not capture.isOpened():
    raise IOError("캡처를 할 수 없습니다")

In [3]:
# model load
weights = "yolov4.weights"
config = "yolov4.cfg"
model = cv2.dnn_DetectionModel(weights, config)

# model에 사용할 엔진과 장치 설정
model.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
model.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

< cv2.dnn.Model 000001CA9CABB950>

In [4]:
# model 파라미터 설정
# Yolo 입력이미지를 0.0~1.0범위로 정규화 1/255.0
scale = 1.0 / 255.0

# 입력크기
#size = (320, 320)
size = (416, 416)
#size = (512, 512)
#size = (608, 608)

# yolo에서는 평균값을 빼주지 않는다
mean = (0.0, 0.0, 0.0)

# swap -> RGB
swap = True

# 자르기
crop = False

model.setInputParams(scale, size, mean, swap, crop)

# NMS(Non-Maximum Suppression)를 클래스별로 처리
model.setNmsAcrossClasses(False)

< cv2.dnn.DetectionModel 000001CA9D08DC10>

In [5]:
# class list <- 객체 인식 종류
classes = read_classes("coco.names")
# color table <- 객체인식 박스 라인의 컬러
colors = get_colors(len(classes))

In [6]:
while True:

    ret, image = capture.read()
    if ret is False:
        cv2.waitKey(0)
        break

    # image 3채널이 아닌 경우 -> 3채널로 변환
    channels = 1 if len(image.shape) == 2 else image.shape[2]
    if channels == 1: # 흑백
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if channels == 4: # rgba
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)

    # 객체 검출    
    confidence_threshold = 0.5  # 신뢰도 임계값    
    nms_threshold = 0.4 # 임계값

    class_ids, confidences, boxes = model.detect(image,
                                                 confidence_threshold,
                                                 nms_threshold)
    '''
    class_ids: 탐지된 객체 클래스 ID 목록  
                           예: [0, 2, 2, 7] → 사람, 자동차, 자동차, 트럭
    confidences: 각 객체의 신뢰도(확률) 목록 
                           예: [0.92, 0.81, 0.77, 0.66]
    boxes: 탐지된 객체의 바운딩 박스 좌표 
                           예: [[x, y, w, h], [x, y, w, h], ...]
    '''

    # 2차원 -> 1차원 배열 변환
    class_ids = np.array(class_ids).flatten()
    confidences = np.array(confidences).flatten()

    # 검출된 객체를 그리기
    for class_id, confidence, box in zip(class_ids, confidences, boxes):
        class_name = classes[class_id]
        color = colors[class_id]
        thickness = 2
        cv2.rectangle(image, box, color, thickness, cv2.LINE_AA)

        result = "{0} ({1:.3f})".format(class_name, confidence)
        point = (box[0], box[1] - 5)
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.5
        cv2.putText(image, result, point, font, scale, color, thickness, cv2.LINE_AA)

    # image 표시
    cv2.imshow("Yolov4 object detection", image)
    key = cv2.waitKey(10)
    if key == ord('q'):
        break

capture.release()
cv2.destroyAllWindows()        